In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sin, cos, lit, year, month, lag, date_format, sum, min, max, col, count, monotonically_increasing_id, row_number, countDistinct, filter
from pyspark.sql.window import Window
import matplotlib.pyplot as plt



# Create or retrieve a SparkSession
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .config("spark.some.config.option", "config-value") \
    .getOrCreate()    

In [0]:
   
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/0725catalog/silver/data/Online Retail SS.csv")\
    .withColumn("CustomerID", col("CustomerID").cast("INT"))
# x InvoiceNo,
# x StockCode,Description, -> map the stockcode to product group
# + Quantity,UnitPrice, -> AMT
# ? InvoiceDate, Are you gonna do Time series anlaysis 
# x CustomerID,
# + Country

df = df.withColumn("year_month", date_format("InvoiceDate", "yyyy-MM"))
# Linear regression to find the how Country explains the AMT 
df = df.withColumn("Amt", col("Quantity") * col("UnitPrice"))



df_monthly = df.groupBy("CustomerID","year_month", "Country").agg(sum('Amt').alias('monthly_spent'))
df_monthly.show()



In [0]:
window_spec = Window.partitionBy("CustomerID").orderBy("year_month")
df_monthly = df_monthly \
    .withColumn("lag_1", lag("monthly_spent", 1).over(window_spec)) \
    .withColumn("lag_2", lag("monthly_spent", 2).over(window_spec)) \
    .withColumn("lag_3", lag("monthly_spent", 3).over(window_spec)) \
    .withColumn("target", lag("monthly_spent", -1).over(window_spec))
    
df_monthly = df_monthly.filter(col("CustomerID").isNotNull()).fillna(0)
df_monthly.display()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, udf
from pyspark.ml.linalg import SparseVector, VectorUDT

class OneHotEncoderRevised:
    def __init__(self, inputCols, outputCols):
        assert len(inputCols) == len(outputCols), "inputCols and outputCols must be same length"
        self.inputCols = inputCols
        self.outputCols = outputCols
        self.categoryMaps = {}

    def fit(self, df: DataFrame):
        for col_name in self.inputCols:
            categories = df.select(col_name).distinct().na.drop().orderBy(col_name).collect()
            categories = [row[0] for row in categories]
            self.categoryMaps[col_name] = {cat: idx for idx, cat in enumerate(categories)}
        return self

    def transform(self, df: DataFrame) -> DataFrame:
        def encode_onehot(categoryMap):
            def encode(value):
                size = len(categoryMap)
                if value not in categoryMap:
                    return SparseVector(size, [], [])
                idx = categoryMap[value]
                return SparseVector(size, [idx], [1.0])
            return encode

        for inputCol, outputCol in zip(self.inputCols, self.outputCols):
            categoryMap = self.categoryMaps[inputCol]
            encode_udf = udf(encode_onehot(categoryMap), VectorUDT())
            df = df.withColumn(outputCol, encode_udf(col(inputCol)))
        return df


In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.sql.functions import dense_rank
# Assemble features
# once you have cat variable
# string index
print(StringIndexer)
#indexer = StringIndexer(inputCol="Country", outputCol="Country_Index")
#indexer = StringIndexer().setInputCol("Country").setOutputCol##("Country_Index")

#df_indexed = indexer.fit(df_monthly).transform(df_monthly)

df_indexed = df_monthly.withColumn(
    "Country_Index",
    dense_rank().over(Window.orderBy("Country")) - 1
)

# onehotencoder
#encoder = OneHotEncoder( inputCols=["Country_Index"], outputCols=["Country_Vec"])

encoder = OneHotEncoderRevised( inputCols=["Country_Index"], outputCols=["Country_Vec"])
df_encoded = encoder.fit(df_indexed).transform(df_indexed)


df_encoded = df_encoded.withColumn("year", year(col("year_month")))
df_encoded = df_encoded.withColumn("month", month(col("year_month")))
df_encoded = df_encoded.withColumn("month_sin", sin(2 * 3.1416 * col("month") / lit(12)))
df_encoded = df_encoded.withColumn("month_cos", cos(2 * 3.1416 * col("month") / lit(12)))

assembler = VectorAssembler(inputCols=["year", "month_sin", "month_cos","Country_Vec", "lag_1", "lag_2", "lag_3"], outputCol="features")
df = assembler.transform(df_encoded)
df.show()
df_sorted = df.orderBy("year_month")

# supervised regression - we need have training and test 
# train_data, test_data = df.randomSplit([0.8, 0.2], seed = 42)

split_point = int(df.count() * 0.8)
train_data = df.limit(split_point)
test_data = df.subtract(train_data)



lr = LinearRegression(featuresCol="features", labelCol="target")
dc = DecisionTreeRegressor(featuresCol="features", labelCol="target")

for r in [lr, dc]:
    model = r.fit(train_data)
    pred = model.transform(test_data)

    evaluator = RegressionEvaluator(
            labelCol="target",
            predictionCol="prediction",
            metricName="rmse"
    )
    e_result = evaluator.evaluate(pred)
    print(f"RMSE: {e_result}")
